## Cell 1: Import all library

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder, LabelEncoder

## Cell 2: Load dataset

In [2]:
df = sns.load_dataset("titanic")
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


## Cell 3:

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB


## Cell 4:

In [4]:
df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

## Cell 5:

In [5]:
df_full = df.copy()
df_full["age"] = df_full["age"].fillna(df_full["age"].mean())
df_full["age"].isnull().sum()

np.int64(0)

## Cell 6: Fill categorycal column using most frequent value

In [6]:
df_full["embarked"] = df_full["embarked"].fillna(df_full["embarked"].mode()[0])
df_full["embarked"].isnull().sum()

np.int64(0)

## Cell 7: Separate numeric and categorical columns

In [7]:
num_cols = df_full.select_dtypes(include=np.number).columns
cat_cols = df_full.select_dtypes(include=["object", "category", "string"]).columns
num_cols, cat_cols

(Index(['survived', 'pclass', 'age', 'sibsp', 'parch', 'fare'], dtype='object'),
 Index(['sex', 'embarked', 'class', 'who', 'deck', 'embark_town', 'alive'], dtype='object'))

## Cell 8: Numeric imputer

In [8]:
num_imputer = SimpleImputer(strategy="median")
df[num_cols] = num_imputer.fit_transform(df[num_cols])

## Cell 9: Categorical imputer

In [9]:
cat_imputer = SimpleImputer(strategy="most_frequent")
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

## Cell 10: Confirm no missing value

In [10]:
df.isnull().sum()

survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
deck           0
embark_town    0
alive          0
alone          0
dtype: int64

## Cell 11: Standard Scaling

In [11]:
df.sample(5)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
541,0.0,3.0,female,9.0,4.0,2.0,31.2750,S,Third,child,False,C,Southampton,no,False
86,0.0,3.0,male,16.0,1.0,3.0,34.3750,S,Third,man,True,C,Southampton,no,False
136,1.0,1.0,female,19.0,0.0,2.0,26.2833,S,First,woman,False,D,Southampton,yes,False
702,0.0,3.0,female,18.0,0.0,1.0,14.4542,C,Third,woman,False,C,Cherbourg,no,False
756,0.0,3.0,male,28.0,0.0,0.0,7.7958,S,Third,man,True,C,Southampton,no,True


In [12]:
scaler = StandardScaler()
scaled = scaler.fit_transform(df[num_cols])
scaled_df = pd.DataFrame(scaled, columns=num_cols)
scaled_df.head()

,survived,pclass,age,sibsp,parch,fare
0,-0.789272,0.827377,-0.565736,0.432793,-0.473674,-0.502445
1,1.266990,-1.566107,0.663861,0.432793,-0.473674,0.786845
2,1.266990,0.827377,-0.258337,-0.474545,-0.473674,-0.488854
3,1.266990,-1.566107,0.433312,0.432793,-0.473674,0.420730
4,-0.789272,0.827377,0.433312,-0.474545,-0.473674,-0.486337


## Min Max Scaler

In [13]:
Minmax = MinMaxScaler()
norm = Minmax.fit_transform(df[num_cols])
norm_df = pd.DataFrame(norm, columns=num_cols)
norm_df.head()

,survived,pclass,age,sibsp,parch,fare
0,0.0,1.0,0.271174,0.125,0.0,0.014151
1,1.0,0.0,0.472229,0.125,0.0,0.139136
2,1.0,1.0,0.321438,0.000,0.0,0.015469
3,1.0,0.0,0.434531,0.125,0.0,0.103644
4,0.0,1.0,0.434531,0.000,0.0,0.015713


## Cell 13: One Hot Encoding

In [14]:
ohe = OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore", dtype=np.int32)
encoded = ohe.fit_transform(df[cat_cols])
encoded_df = pd.DataFrame(encoded, columns=ohe.get_feature_names_out(cat_cols))

display(df[cat_cols].head())
display(encoded_df.head())

,sex,embarked,class,who,deck,embark_town,alive
0,male,S,Third,man,C,Southampton,no
1,female,C,First,woman,C,Cherbourg,yes
2,female,S,Third,woman,C,Southampton,yes
3,female,S,First,woman,C,Southampton,yes
4,male,S,Third,man,C,Southampton,no


,sex_male,embarked_Q,embarked_S,class_Second,class_Third,who_man,who_woman,deck_B,deck_C,deck_D,deck_E,deck_F,deck_G,embark_town_Queenstown,embark_town_Southampton,alive_yes
0,1,0,1,0,1,1,0,0,1,0,0,0,0,0,1,0
1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1
2,0,0,1,0,1,0,1,0,1,0,0,0,0,0,1,1
3,0,0,1,0,0,0,1,0,1,0,0,0,0,0,1,1
4,1,0,1,0,1,1,0,0,1,0,0,0,0,0,1,0


In [15]:
ord_enccoder = OrdinalEncoder()
ord_data = ord_enccoder.fit_transform(df[['class']])
ord_df = pd.DataFrame(ord_data, columns=['class'])
ord_df.head(10)

,class
0,2.0
1,0.0
2,2.0
3,0.0
4,2.0
5,2.0
6,0.0
7,2.0
8,2.0
9,1.0


In [16]:
data = {
    "Name": ["Alice", "Bob", "Charlie", "David", "Eve"],
    "Result": ["Pass", "Fail", "Pass", "Fail", "Pass"]
}

df_samll = pd.DataFrame(data)
print("Original DataFrame: ")
display(df_samll)

le = LabelEncoder()
df_samll["Result_Encoded"] = le.fit_transform(df_samll["Result"])
print("\nAfter Label Encoding: ")
display(df_samll)

Original DataFrame: 


,Name,Result
0,Alice,Pass
1,Bob,Fail
2,Charlie,Pass
3,David,Fail
4,Eve,Pass



After Label Encoding: 


,Name,Result,Result_Encoded
0,Alice,Pass,1
1,Bob,Fail,0
2,Charlie,Pass,1
3,David,Fail,0
4,Eve,Pass,1
